# AsyncGenerator 用法（结合 HR Agent 场景）

`async def` + `yield` = 异步生成器：**可以 await 的惰性事件流**。
本项目里 LLM 流式输出、旧流式链路（progress/delta/final）都是它。

## 1. 基础：创建不执行，迭代才吐值

In [1]:
async def count():
    for i in range(3):
        print(f"  生产 {i}")
        yield i

agen = count()          # 此刻函数体一行都没跑
print("创建后:", agen)
async for x in agen:    # 每次 await 到下一个 yield
    print("消费", x)

创建后: <async_generator object count at 0x109bd5f20>
  生产 0
消费 0
  生产 1
消费 1
  生产 2
消费 2


## 2. 核心能力：yield 之间可以 await（模拟 LLM 逐 token 输出）

In [2]:
import asyncio
from typing import AsyncGenerator

async def fake_llm_stream(prompt: str) -> AsyncGenerator[str, None]:
    for token in ["你好", "，我是", "HR", "助手"]:
        await asyncio.sleep(0.9)   # 普通 Generator 做不到这一步
        yield token
gen = fake_llm_stream("hi")
print(gen)
async for delta in gen:
    print(delta, flush=True)
print()

<async_generator object fake_llm_stream at 0x109bd7c60>
你好
，我是
HR
助手



## 3. 项目事件流协议：progress / delta / final

旧流式链路（如 `stream_recruitment_agent`）按此协议吐事件，对应 SSE 推送。

In [3]:
from typing import Any, Dict

async def hr_event_stream() -> AsyncGenerator[Dict[str, Any], None]:
    yield {"type": "progress", "step": "解析需求"}
    for token in ["Java", "开发", "工程师", "AI ","大模型应用","开发","AI ","Agent Dev"]:
        await asyncio.sleep(0.9)
        yield {"type": "delta", "delta": token}
    yield {"type": "final", "response": {"message": "Java开发工程师", "intent": "jd"}}

async for event in hr_event_stream():
    print(event["type"], "->", event)

progress -> {'type': 'progress', 'step': '解析需求'}
delta -> {'type': 'delta', 'delta': 'Java'}
delta -> {'type': 'delta', 'delta': '开发'}
delta -> {'type': 'delta', 'delta': '工程师'}
delta -> {'type': 'delta', 'delta': 'AI '}
delta -> {'type': 'delta', 'delta': '大模型应用'}
delta -> {'type': 'delta', 'delta': '开发'}
delta -> {'type': 'delta', 'delta': 'AI '}
delta -> {'type': 'delta', 'delta': 'Agent Dev'}
final -> {'type': 'final', 'response': {'message': 'Java开发工程师', 'intent': 'jd'}}


## 4. 真实项目函数：`_emit_generator`（legacy.py:66）

非 final 事件透传给 stream writer，final 拦截后返回。
`get_stream_writer` 依赖图运行时，这里临时换成列表记录（同单测做法）。

In [4]:
import os, sys
from pathlib import Path
BACKEND = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
os.chdir(BACKEND); sys.path.insert(0, str(BACKEND))
from app.core.config import settings
settings.DEBUG = False

from app.agent.nodes import legacy

sent = []
original = legacy.get_stream_writer
legacy.get_stream_writer = lambda: sent.append   # writer(event) == sent.append(event)
try:
    final = await legacy._emit_generator(hr_event_stream())
finally:
    legacy.get_stream_writer = original

print("透传给 writer:", sent)
print("返回的 final:", final)
assert final["type"] == "final" and len(sent) == 9   # final 不透传

透传给 writer: [{'agent_event': {'type': 'progress', 'step': '解析需求'}}, {'agent_event': {'type': 'delta', 'delta': 'Java'}}, {'agent_event': {'type': 'delta', 'delta': '开发'}}, {'agent_event': {'type': 'delta', 'delta': '工程师'}}, {'agent_event': {'type': 'delta', 'delta': 'AI '}}, {'agent_event': {'type': 'delta', 'delta': '大模型应用'}}, {'agent_event': {'type': 'delta', 'delta': '开发'}}, {'agent_event': {'type': 'delta', 'delta': 'AI '}}, {'agent_event': {'type': 'delta', 'delta': 'Agent Dev'}}]
返回的 final: {'type': 'final', 'response': {'message': 'Java开发工程师', 'intent': 'jd'}}


## 5. 真实 LLM 流式：`LLMService.stream_response`（chat_node 同款）

In [6]:
from app.services.llm_service import LLMService

llm = LLMService()
async for delta in llm.stream_response("用200个字左右介绍什么是岗位JD"):
    print(delta, end="", flush=True)
print()

岗位JD（Job Description，岗位说明书）是企业对某一职位的正式描述，用于明确岗位存在的目的、主要职责、工作关系和任职条件。它通常包含岗位名称、所属部门、汇报对象、核心任务、任职资格、所需技能、学历经验要求，以及工作地点、薪酬福利等信息。对招聘而言，JD是发布职位、筛选简历、设计面试问题和评估候选人的基础依据；对员工而言，它有助于理解岗位职责与绩效期望。
